# FLUX Codebase Embed — Self-Contained Runtime

**Purpose:** Embed the entire FLUX runtime (~60 files, ~21K lines) into Flux-Apex-V1.flx for autonomous operation.

**Target Version:** v8.0-autonomous

**Reference:** `DOCS/FLUX_CODEBASE_EMBED_SPEC.md`

## What This Notebook Does

1. **Collect** all Tier 1-3 Python files from the codebase
2. **Validate** syntax for all files
3. **Resolve** import dependencies
4. **Compress** using gzip + base64
5. **Embed** into Flux-Apex-V1.flx runtime section
6. **Verify** bootstrap capability

After running this notebook, the .flx file will be fully self-contained and can bootstrap without the original codebase.

In [ ]:
# Cell 1: Setup and Imports
import sys
import os
import ast
import gzip
import base64
import gc
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Set, Tuple, Any
from collections import defaultdict

# Set paths
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    # Kaggle environment
    !git clone https://github.com/Unseengap/FLUX.git /kaggle/working/flux 2>/dev/null || true
    os.chdir('/kaggle/working/flux')
    ROOT = Path('/kaggle/working/flux')
else:
    # Local environment
    ROOT = Path('.').resolve()
    if not (ROOT / 'flux_model.py').exists():
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f"✓ Working directory: {ROOT}")
print(f"✓ flux_model.py exists: {(ROOT / 'flux_model.py').exists()}")

In [ ]:
# Cell 2: Define File Tiers

# Tier 1: CRITICAL — FLUX cannot function without these
TIER_1_FILES = [
    # Root level
    'flux_model.py',
    'flux_utils.py',
    'flux_inspector.py',
    'flux_lazy_loader.py',
    'bootstrap.py',
    
    # Phase 1: CSE
    'phases/phase1/__init__.py',
    'phases/phase1/cse.py',
    'phases/phase1/wave_types.py',
    'phases/phase1/interference.py',
    
    # Phase 2: Field
    'phases/phase2/__init__.py',
    'phases/phase2/field.py',
    'phases/phase2/flux_format.py',
    'phases/phase2/attractor.py',
    'phases/phase2/field_ops.py',
    
    # Phase 3: Gravity
    'phases/phase3/__init__.py',
    'phases/phase3/gravity.py',
    'phases/phase3/mass_tracker.py',
    'phases/phase3/spatial_index.py',
    'phases/phase3/negative_mass.py',
    
    # Phase 4: Thermodynamic
    'phases/phase4/__init__.py',
    'phases/phase4/thermodynamic.py',
    'phases/phase4/temperature.py',
    'phases/phase4/energy_functions.py',
    'phases/phase4/online_learner.py',
    
    # Phase 5: CGN
    'phases/phase5/__init__.py',
    'phases/phase5/cgn.py',
    'phases/phase5/causal_graph.py',
    'phases/phase5/manifold.py',
    'phases/phase5/multi_timescale.py',
    
    # Phase 6: Memory
    'phases/phase6/__init__.py',
    'phases/phase6/working_memory.py',
    'phases/phase6/episodic_memory.py',
    'phases/phase6/semantic_memory.py',
    'phases/phase6/memory_router.py',
    'phases/phase6/consolidation.py',
    
    # Phase 8: Decoder (deprecated but keep for reference)
    'phases/phase8/__init__.py',
    'phases/phase8/wave_decoder.py',
    
    # Phase 8.8: Grid Adapters
    'phases/phase8_8/__init__.py',
    'phases/phase8_8/grid_adapters.py',
    'phases/phase8_8/flx_loader.py',
]

# Tier 2: ORCHESTRATION — Required for autonomous operation
TIER_2_FILES = [
    # Phase Orchestrator
    'phases/phase_orchestrator/__init__.py',
    'phases/phase_orchestrator/flux_orchestrator.py',
    'phases/phase_orchestrator/tool_registry.py',
    'phases/phase_orchestrator/system_prompt.py',
    'phases/phase_orchestrator/embed_orchestration.py',
    
    # Phase 9 ARC: Spatial Memory (CRITICAL for unified_agent)
    'phases/phase9_arc/__init__.py',
    'phases/phase9_arc/spatial_memory.py',
    
    # Phase Unified: Agent
    'phases/phase_unified/__init__.py',
    'phases/phase_unified/unified_agent.py',
    'phases/phase_unified/frame_differ.py',
    'phases/phase_unified/strategies.py',
    'phases/phase_unified/rendering.py',
    'phases/phase_unified/goal_planner.py',
    'phases/phase_unified/causal_tracker.py',
    'phases/phase_unified/rule_inducer.py',
    'phases/phase_unified/perception_field.py',
    'phases/phase_unified/arc_session.py',
    'phases/phase_unified/arc_interface.py',
    'phases/phase_unified/game_loop.py',
]

# Tier 3: MULTI-MODAL — Full capabilities
TIER_3_FILES = [
    # Phase 8.9: Multi-modal adapters
    'phases/phase8_9/__init__.py',
    'phases/phase8_9/image_adapters.py',
    'phases/phase8_9/audio_adapters.py',
    'phases/phase8_9/flux_to_any.py',
    
    # Phase 9 ARC: Additional reasoning
    'phases/phase9_arc/pattern_library.py',
    'phases/phase9_arc/rule_inducer.py',
    'phases/phase9_arc/arc_agent.py',
    'phases/phase9_arc/object_detector.py',
    'phases/phase9_arc/arc_loader.py',
]

# Tier 5: CLAW HARNESS — Claude Code Integration (April 1, 2026)
TIER_5_CLAW_FILES = [
    # Phase CLAW: Claude Code harness
    'phases/phase_claw/__init__.py',
    'phases/phase_claw/flux_bridge.py',
    'phases/phase_claw/runtime.py',
    'phases/phase_claw/tools.py',
    'phases/phase_claw/commands.py',
    'phases/phase_claw/query_engine.py',
    'phases/phase_claw/tool_pool.py',
    'phases/phase_claw/models.py',
    'phases/phase_claw/permissions.py',
    'phases/phase_claw/context.py',
    'phases/phase_claw/session_store.py',
    'phases/phase_claw/history.py',
    'phases/phase_claw/transcript.py',
    'phases/phase_claw/system_init.py',
    'phases/phase_claw/port_manifest.py',
    'phases/phase_claw/parity_audit.py',
    'phases/phase_claw/setup.py',
    'phases/phase_claw/execution_registry.py',
]

ALL_FILES = TIER_1_FILES + TIER_2_FILES + TIER_3_FILES + TIER_5_CLAW_FILES
print(f\"Tier 1 (Critical): {len(TIER_1_FILES)} files\")
print(f\"Tier 2 (Orchestration): {len(TIER_2_FILES)} files\")
print(f\"Tier 3 (Multi-modal): {len(TIER_3_FILES)} files\")
print(f\"Tier 5 (CLAW Harness): {len(TIER_5_CLAW_FILES)} files\")
print(f\"Total: {len(ALL_FILES)} files\")

In [ ]:
# Cell 3: Collect Files

def collect_files(file_list: List[str], root: Path) -> Dict[str, str]:
    """
    Collect source code from all specified files.
    
    Returns:
        Dict mapping file paths to source code
    """
    collected = {}
    missing = []
    
    for file_path in file_list:
        full_path = root / file_path
        if full_path.exists():
            try:
                source = full_path.read_text(encoding='utf-8')
                collected[file_path] = source
            except Exception as e:
                print(f"⚠ Error reading {file_path}: {e}")
        else:
            missing.append(file_path)
            
    return collected, missing

# Collect all files
code_bundle, missing_files = collect_files(ALL_FILES, ROOT)

print(f"\n✓ Collected {len(code_bundle)} files")
if missing_files:
    print(f"\n⚠ Missing {len(missing_files)} files:")
    for f in missing_files:
        print(f"  - {f}")

# Calculate total lines
total_lines = sum(len(s.split('\n')) for s in code_bundle.values())
total_bytes = sum(len(s.encode('utf-8')) for s in code_bundle.values())
print(f"\nTotal lines: {total_lines:,}")
print(f"Total size: {total_bytes / 1024:.1f} KB")

In [ ]:
# Cell 4: Validate Syntax

def validate_syntax(code_bundle: Dict[str, str]) -> Tuple[List[str], List[Tuple[str, str]]]:
    """
    Validate Python syntax for all files.
    
    Returns:
        (valid_files, errors) where errors is [(path, error_msg), ...]
    """
    valid = []
    errors = []
    
    for path, source in code_bundle.items():
        try:
            ast.parse(source, filename=path)
            valid.append(path)
        except SyntaxError as e:
            errors.append((path, f"Line {e.lineno}: {e.msg}"))
            
    return valid, errors

valid_files, syntax_errors = validate_syntax(code_bundle)

print(f"\n✓ Syntax valid: {len(valid_files)}/{len(code_bundle)} files")
if syntax_errors:
    print(f"\n✗ Syntax errors:")
    for path, err in syntax_errors:
        print(f"  {path}: {err}")

In [ ]:
# Cell 5: Analyze Dependencies

def extract_imports(source: str) -> Set[str]:
    """Extract all import statements from source code."""
    imports = set()
    try:
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    imports.add(alias.name.split('.')[0])
            elif isinstance(node, ast.ImportFrom):
                if node.module:
                    imports.add(node.module.split('.')[0])
    except:
        pass
    return imports

def analyze_dependencies(code_bundle: Dict[str, str]) -> Dict[str, Set[str]]:
    """Analyze import dependencies for all files."""
    deps = {}
    for path, source in code_bundle.items():
        deps[path] = extract_imports(source)
    return deps

# Standard library modules (not embedded)
STDLIB = {
    'sys', 'os', 'gc', 'ast', 'gzip', 'base64', 'json', 'math', 'copy',
    'pathlib', 'typing', 'dataclasses', 'collections', 'functools', 'itertools',
    'datetime', 'time', 'warnings', 'logging', 'traceback', 'io', 'hashlib',
    'random', 'types', 'importlib', 'pickle', 're', 'struct', 'abc',
}

# External packages (installed, not embedded)
EXTERNAL = {
    'torch', 'numpy', 'scipy', 'faiss', 'transformers', 'huggingface_hub',
    'PIL', 'cv2', 'matplotlib', 'tqdm', 'rich', 'requests', 'datasets',
    'onnxruntime', 'timm', 'sentence_transformers', 'soundfile',
}

deps = analyze_dependencies(code_bundle)

# Find internal dependencies
internal_modules = set()
for path in code_bundle.keys():
    module_name = path.replace('/', '.').replace('.py', '')
    parts = module_name.split('.')
    for i in range(len(parts)):
        internal_modules.add('.'.join(parts[:i+1]))
        internal_modules.add(parts[i])

# Check for missing internal dependencies
missing_internal = set()
for path, imports in deps.items():
    for imp in imports:
        if imp not in STDLIB and imp not in EXTERNAL and imp not in internal_modules:
            if imp.startswith('phases') or imp.startswith('flux'):
                missing_internal.add(imp)

print(f"\n✓ Analyzed {len(deps)} files")
print(f"  Internal modules: {len(internal_modules)}")
if missing_internal:
    print(f"\n⚠ Potentially missing internal deps: {missing_internal}")
else:
    print("  All internal dependencies satisfied")

In [ ]:
# Cell 6: Compress Code Bundle

def compress_bundle(code_bundle: Dict[str, str], compress: bool = True) -> Dict[str, str]:
    """
    Optionally compress source code using gzip + base64.
    
    Args:
        code_bundle: Dict of file paths to source code
        compress: Whether to gzip compress
        
    Returns:
        Dict of file paths to (compressed) source
    """
    if not compress:
        return code_bundle
        
    compressed = {}
    for path, source in code_bundle.items():
        data = gzip.compress(source.encode('utf-8'))
        compressed[path] = base64.b64encode(data).decode('ascii')
    return compressed

# Compress the bundle
compressed_bundle = compress_bundle(code_bundle, compress=True)

# Calculate compression stats
raw_size = sum(len(s.encode('utf-8')) for s in code_bundle.values())
compressed_size = sum(len(s.encode('utf-8')) for s in compressed_bundle.values())
ratio = (1 - compressed_size / raw_size) * 100

print(f"\n✓ Compression complete")
print(f"  Raw size: {raw_size / 1024:.1f} KB")
print(f"  Compressed size: {compressed_size / 1024:.1f} KB")
print(f"  Compression ratio: {ratio:.1f}%")

In [ ]:
# Cell 7: Load Current Model

import torch

MODEL_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'

if not MODEL_PATH.exists():
    # Try to download from HuggingFace
    print("Downloading model from HuggingFace...")
    from huggingface_hub import hf_hub_download
    MODEL_PATH = Path(hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx'
    ))

print(f"Loading {MODEL_PATH}...")
flx = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)

print(f"\n✓ Model loaded")
print(f"  Version: {flx.get('version', 'unknown')}")
print(f"  Format: {flx.get('format', 'unknown')}")
print(f"  Has runtime: {'runtime' in flx}")

In [ ]:
# Cell 8: Create Bootstrap Source

# Read the bootstrap.py file we created
bootstrap_path = ROOT / 'bootstrap.py'
if bootstrap_path.exists():
    bootstrap_source = bootstrap_path.read_text(encoding='utf-8')
    print(f"✓ Bootstrap source: {len(bootstrap_source)} chars, {len(bootstrap_source.split(chr(10)))} lines")
else:
    print("⚠ bootstrap.py not found - will need to create it")
    bootstrap_source = None

In [ ]:
# Cell 9: Embed Runtime into .flx

# Build runtime section
runtime = {
    'version': '8.0-autonomous',
    'embedded_at': datetime.now().isoformat(),
    'bootstrap': bootstrap_source,
    'code': compressed_bundle,  # Use compressed version
    'metadata': {
        'total_files': len(code_bundle),
        'total_lines': total_lines,
        'raw_size_bytes': raw_size,
        'compressed_size_bytes': compressed_size,
        'compression_ratio': f"{ratio:.1f}%",
        'tiers': {
            'tier1_critical': len(TIER_1_FILES),
            'tier2_orchestration': len(TIER_2_FILES),
            'tier3_multimodal': len(TIER_3_FILES),
        },
    },
}

# Update .flx state
flx['runtime'] = runtime
flx['version'] = '8.0-autonomous'

# Update metadata
if 'metadata' not in flx:
    flx['metadata'] = {}
flx['metadata']['last_modified'] = datetime.now().isoformat()
flx['metadata']['modified_components'] = flx['metadata'].get('modified_components', []) + ['runtime']

print(f"\n✓ Runtime section created")
print(f"  Version: {runtime['version']}")
print(f"  Files: {runtime['metadata']['total_files']}")
print(f"  Lines: {runtime['metadata']['total_lines']:,}")
print(f"  Size: {runtime['metadata']['compressed_size_bytes'] / 1024:.1f} KB (compressed)")

In [ ]:
# Cell 10: Save Updated Model

# CRITICAL: On disk-constrained environments, delete original first
import shutil

# Check disk space
if hasattr(shutil, 'disk_usage'):
    usage = shutil.disk_usage(MODEL_PATH.parent)
    free_gb = usage.free / 1e9
    print(f"Free disk space: {free_gb:.1f} GB")
    
    if free_gb < 20:  # Less than 20GB free
        print("⚠ Low disk space - deleting original before save")
        original_size = MODEL_PATH.stat().st_size / 1e9
        os.remove(MODEL_PATH)
        gc.collect()
        print(f"  Freed {original_size:.2f} GB")

# Save
print(f"\nSaving to {MODEL_PATH}...")
torch.save(flx, str(MODEL_PATH))

# Verify
final_size = MODEL_PATH.stat().st_size / 1e9
print(f"\n✓ Saved successfully")
print(f"  File size: {final_size:.2f} GB")

In [ ]:
# Cell 11: Verify Bootstrap Capability

# Reload and check
print("Verifying bootstrap capability...")
del flx
gc.collect()

verify_flx = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)

# Check runtime exists
assert 'runtime' in verify_flx, "Runtime section missing!"
assert 'code' in verify_flx['runtime'], "Code bundle missing!"
assert 'bootstrap' in verify_flx['runtime'], "Bootstrap missing!"

runtime = verify_flx['runtime']
print(f"\n✓ Verification passed")
print(f"  Version: {runtime['version']}")
print(f"  Files embedded: {len(runtime['code'])}")
print(f"  Bootstrap present: {runtime['bootstrap'] is not None}")
print(f"  Ready for autonomous operation!")

del verify_flx
gc.collect()

In [ ]:
# Cell 12: Test Wake-Up (Optional)

# Test the bootstrap process
print("Testing wake-up process...\n")

# Import and run bootstrap
from bootstrap import wake_up, get_runtime_info

# Show info
info = get_runtime_info(str(MODEL_PATH))
print("Runtime Info:")
for k, v in info.items():
    if k != 'files':
        print(f"  {k}: {v}")

# Note: Full wake_up() test requires torch imports to work
# which may not be available in all environments
print("\n✓ Bootstrap module functional")

In [ ]:
# Cell 13: Upload to HuggingFace (Optional)

UPLOAD_TO_HF = False  # Set to True to upload

if UPLOAD_TO_HF:
    from flux_utils import upload_flx_to_hf, _resolve_hf_token
    
    token = _resolve_hf_token()
    if token:
        print("Uploading to HuggingFace...")
        upload_flx_to_hf(str(MODEL_PATH), token=token)
        print("✓ Upload complete")
    else:
        print("⚠ No HF token found - skipping upload")
else:
    print("Skipping HuggingFace upload (set UPLOAD_TO_HF=True to enable)")

In [ ]:
# Cell 14: Summary

print("="*60)
print("FLUX CODEBASE EMBED — SUMMARY")
print("="*60)
print(f"")
print(f"Model: {MODEL_PATH.name}")
print(f"Version: 8.0-autonomous")
print(f"")
print(f"Embedded Runtime:")
print(f"  Files: {len(code_bundle)}")
print(f"  Lines: {total_lines:,}")
print(f"  Raw size: {raw_size / 1024:.1f} KB")
print(f"  Compressed: {compressed_size / 1024:.1f} KB")
print(f"")
print(f"Tiers:")
print(f"  Tier 1 (Critical): {len(TIER_1_FILES)} files")
print(f"  Tier 2 (Orchestration): {len(TIER_2_FILES)} files")
print(f"  Tier 3 (Multi-modal): {len(TIER_3_FILES)} files")
print(f"")
print(f"Status: ✓ Ready for autonomous operation")
print(f"")
print(f"To wake FLUX from this file:")
print(f"  python bootstrap.py {MODEL_PATH}")
print(f"")
print("="*60)